# 04 — MC1 schema and cross-year stability

Identify MC1-only structurally unavailable attributes, retain SMART 211, and verify whether the dominant MC1 null pattern changes across years.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPORT_DIR = Path("../exports")
assert EXPORT_DIR.exists(), f"Expected exports directory at {EXPORT_DIR.resolve()}"

def load_csv(name):
    path = EXPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing required export: {path}")
    return pd.read_csv(path)

def drop_export_index(df):
    return df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")], errors="ignore")

def attr_id(variable):
    return int(str(variable).split("_")[-1])

def canonical_pattern(serialized):
    return tuple(sorted(json.loads(serialized)))

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)


In [2]:
a18=load_csv("smart_2018_mc1_feature_availability.csv")
a19=load_csv("smart_2019_mc1_feature_availability.csv")
s19=drop_export_index(load_csv("smart_2019_mc1_structurally_unavailable.csv"))
p18=load_csv("smart_2018_mc1_null_patterns.csv")
p19=load_csv("smart_2019_mc1_null_patterns.csv")

## MC1 structurally unavailable attributes

In [3]:
ids18=sorted({attr_id(v) for v in a18.VARIABLE})
ids19=sorted({attr_id(v) for v in s19.VARIABLE})
print("2018 candidate attributes:",ids18)
print("2019 candidate attributes:",ids19)
print("Shared MC1 structural-unavailability set:", sorted(set(ids18)&set(ids19)))
assert set(ids18)==set(ids19)

2018 candidate attributes: [175, 177, 181, 182, 190, 192, 232, 233, 241, 242, 244, 245]
2019 candidate attributes: [175, 177, 181, 182, 190, 192, 232, 233, 241, 242, 244, 245]
Shared MC1 structural-unavailability set: [175, 177, 181, 182, 190, 192, 232, 233, 241, 242, 244, 245]


## SMART 211 in MC1 during 2019

In [4]:
display(a19[a19.ATTRIBUTE_ID.eq(211)])

,ATTRIBUTE_ID,R_COLUMN,R_NULL_COUNT,R_NON_NULL_COUNT,R_NULL_PERCENT,N_COLUMN,N_NULL_COUNT,N_NON_NULL_COUNT,N_NULL_PERCENT,SAME_MISSINGNESS_COUNT,SAME_MISSINGNESS_PERCENT,MISSINGNESS_ALWAYS_MATCHES
29,211,R_211,61481858,2738,99.995547,N_211,61481858,2738,99.995547,61484596,100.0,True


## Dominant pattern stability

In [5]:
top18=p18.sort_values("PATTERN_RANK").iloc[0]
top19=p19.sort_values("PATTERN_RANK").iloc[0]
comparison=pd.DataFrame([
    [2018,top18.NULL_PATTERN_ID,int(top18.ROW_COUNT),int(top18.MC1_ROWS),top18.MC1_PERCENT],
    [2019,top19.NULL_PATTERN_ID,int(top19.ROW_COUNT),int(top19.MC1_ROWS),top19.MC1_PERCENT]
],columns=["YEAR","NULL_PATTERN_ID","ROW_COUNT","MC1_ROWS","MC1_PERCENT"])
display(comparison)
print("Same dominant pattern:", top18.NULL_PATTERN_ID==top19.NULL_PATTERN_ID)

,YEAR,NULL_PATTERN_ID,ROW_COUNT,MC1_ROWS,MC1_PERCENT
0,2018,d237e8c2e2409dd8e304d0e30a9928d96ed499ac595f6e4c132ff2232474423e,53660875,53789530,99.760818
1,2019,d237e8c2e2409dd8e304d0e30a9928d96ed499ac595f6e4c132ff2232474423e,61469043,61484596,99.974704


Same dominant pattern: True


## Long-tail size

In [6]:
for year,d in [(2018,p18),(2019,p19)]:
    display(d.sort_values("PATTERN_RANK").head(10)[["PATTERN_RANK","ROW_COUNT","MC1_PERCENT","CUMULATIVE_MC1_PERCENT"]].assign(YEAR=year))

,PATTERN_RANK,ROW_COUNT,MC1_PERCENT,CUMULATIVE_MC1_PERCENT,YEAR
0,1,53660875,99.760818,99.760818,2018
1,2,114847,0.213512,99.974330,2018
2,3,859,0.001597,99.975927,2018
3,4,840,0.001562,99.977488,2018
4,5,409,0.000760,99.978249,2018
5,6,387,0.000719,99.978968,2018
6,7,266,0.000495,99.979463,2018
7,8,258,0.000480,99.979942,2018
8,9,247,0.000459,99.980401,2018
9,10,230,0.000428,99.980829,2018


,PATTERN_RANK,ROW_COUNT,MC1_PERCENT,CUMULATIVE_MC1_PERCENT,YEAR
0,1,61469043,99.974704,99.974704,2019
1,2,2738,0.004453,99.979157,2019
2,3,909,0.001478,99.980636,2019
3,4,729,0.001186,99.981821,2019
4,5,404,0.000657,99.982479,2019
5,6,377,0.000613,99.983092,2019
6,7,259,0.000421,99.983513,2019
7,8,243,0.000395,99.983908,2019
8,9,236,0.000384,99.984292,2019
9,10,231,0.000376,99.984668,2019


**Interpretation.** MC1 does not show a wholesale schema change between years. One exact pattern accounts for virtually the entire MC1 population in both years; cross-year changes are concentrated in the rare tail.